In [ ]:
import anndata
import pandas as pd
import numpy as np
from scipy.sparse import issparse
import re

In [ ]:
df_005 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_0.05uM.csv", index_col=[0,1])
df_05 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_0.5uM.csv", index_col=[0,1])
df_50 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_5.0uM.csv", index_col=[0,1])

In [ ]:
dmso_005 = df_005[df_005.index.get_level_values('drug').str.contains('DMSO', na=False)]
dmso_05 = df_05[df_05.index.get_level_values('drug').str.contains('DMSO', na=False)]
dmso_50 = df_50[df_50.index.get_level_values('drug').str.contains('DMSO', na=False)]

In [ ]:
dmso_baseline_005 = {cell: row for (drug, cell), row in dmso_005.iterrows()}
dmso_baseline_05 = {cell: row for (drug, cell), row in dmso_05.iterrows()}
dmso_baseline_50 = {cell: row for (drug, cell), row in dmso_50.iterrows()}

In [ ]:
def subtract_dmso(df, dmso_baseline, conc_name):
    results = []
    for (drug, cell), row in df.iterrows():
        if 'DMSO' in drug:
            continue
        if cell in dmso_baseline:
            diff = row - dmso_baseline[cell]
            results.append([drug, cell] + diff.tolist())
    
    result_df = pd.DataFrame(results, columns=['drug', 'cell'] + list(df.columns))
    result_df = result_df.set_index(['drug', 'cell'])
    return result_df

diff_005 = subtract_dmso(df_005, dmso_baseline_005, '0.05uM')
diff_05 = subtract_dmso(df_05, dmso_baseline_05, '0.5uM')
diff_50 = subtract_dmso(df_50, dmso_baseline_50, '5.0uM')

In [ ]:
diff_005.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_diff_0.05uM.csv")
diff_05.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_diff_0.5uM.csv")
diff_50.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_diff_5.0uM.csv")

In [ ]:
gene_means = pd.concat([
    diff_005.abs().mean(),
    diff_05.abs().mean(),
    diff_50.abs().mean()
]).groupby(level=0).mean()
top_genes = gene_means.sort_values(ascending=False).head(200).index.tolist()

In [ ]:
diff_005 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_0.05uM.csv", index_col=[0,1])
diff_05 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_0.5uM.csv", index_col=[0,1])
diff_50 = pd.read_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_5.0uM.csv", index_col=[0,1])

In [ ]:
diff_005_top = diff_005[top_genes]
diff_05_top = diff_05[top_genes]
diff_50_top = diff_50[top_genes]

In [ ]:
diff_005_top.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_0.05uM_top200.csv")
diff_05_top.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_0.5uM_top200.csv")
diff_50_top.to_csv("/jdata/qmy/VirtualCell/group_mean_drug/drug_cell_diff_5.0uM_top200.csv")